# TQEC - Tutorial

The purpose of this notebook is to introduce the basics of the Surface Code and to illustrate the usage of TQEC as a compiler for a Surface Code quantum processor.

In [ ]:
import numpy
import stim

import sinter
import matplotlib.pyplot as plt
from IPython.core.display import Markdown

from tqec.compile.convention import FIXED_BULK_CONVENTION
from tqec.gallery import cnot
from tqec.simulation import start_simulation_using_sinter
from tqec.simulation.generation import generate_stim_circuits_with_detectors
from tqec.simulation.plotting.inset import plot_threshold_as_inset
from tqec.simulation.threshold import binary_search_threshold
from docs.mqsf.utils_simulation import simulate, sample, make_noisy

from tqec.gallery.memory import memory
from tqec import compile_block_graph, Basis, NoiseModel

## A simple memory experiment

The simplest experiment we can perform on a quantum computer is a memory experiment; prepare a qubit in the X-basis (i.e. state $|{\tt +}\rangle$), let it idle for a while and then measure in the X-basis. We can describe this experiment as a circuit in STIM in the following way:

In [ ]:
experiment = "X-basis memory experiment"
scenario = "Physical"
circuit = stim.Circuit("""
    RX 0
    TICK
    MX 0
    OBSERVABLE_INCLUDE(0) rec[-1]
""")
scenarios = {}
scenarios[scenario] = circuit
display(Markdown(f"[Open in Crumble ({scenario})]({circuit.to_crumble_url()})"))

In [ ]:
circuit.diagram("timeline-svg")

Executing this circuit would tell us what the expected outcome is in the ideal case. But in fault-tolerant computing, we are interested in the effects that noise and errors have on the outcome of our computation. In order to make this circuit more useful, we can introduce noise by adding errors at various points, subject to a probability of occurring.

The Circuit-Level Noise Model (a.k.a. Depolarising Noise Model) assumes a uniform probability with the following patterns of errors
* A preparation may produce an orthogonal state (e.g. prepare $|{\tt +}\rangle$, end up with $|{\tt -}\rangle$)
* A measurement may produce a flipped outcome (e.g. measure $|{\tt +}\rangle$, get a bit ${\tt 1}$; eigenvalue $-1$)
* A 1-qubit gate can be followed by a Pauli-type error (i.e. ${\bf X}$, ${\bf Y}$ or ${\bf Z}$)
* A 2-qubit gate can be followed by an error drawn from $\{{\bf I}, {\bf X}, {\bf Y}, {\bf Z}\}^{\otimes 2} ~\backslash~ \{{\bf I}^{\otimes 2}\}$

In [ ]:
noisy = make_noisy(circuit)
noisy.diagram("timeline-svg")

In order to assess the ability of our computation to produce a correct outcome, we can simulate its behavior considering decreasing values of physical error rates. In fault-tolerant quantum computing, we are interested in understanding the relationship between two error rates;

* the physical error rate (PER) is an aggregate of how noisy the physical qubits and gates are. Reducing this rate requires efforts in engineering and manufacturing processes.
* the logical error rate (LER) is an estimation of how many errors we can expect would occur that would compromise our computation.

In [ ]:
simulate(scenarios, title=experiment, label="Case")

On this curve, we can see a linear relationship between the physical error rate (PER) and the logical error rate (LER);

* in order to achieve a logical error rate of $10^{-2}$, we need a physical error rate of $\approx 5 \times 10^{-3}$
* in order to achieve a logical error rate of $10^{-4}$, we need a physical error rate of $\approx 5 \times 10^{-5}$

This is the behavior you can  expect when using the physical qubits to directly encode your logical qubits (i.e. 1-to-1). In this case, the only way to improve the error rates of your computation is to lower the physical error rate. If our computation contains $10^{9}$ logical operations, we need to have a physical error rate of $10^{-9}$ to be able to run that computation without any error corrupting its execution.

The central proposition behind fault-tolerant quantum computing is that we can leverage a decent physical error rate, say between $10^{-2}$ and $10^{-3}$, but amplify the ability to suppress errors by using groups of physical qubits to represent individual logical qubits. The logical information is encoded onto physical qubits using a specific quantum code, along with the way to perform computations on that logical information. Scalability of current architectures becomes the challenge together with designing quantum codes that offer powerful suppression capabilities.

### How to protect this experiment ?

In TQEC, a fault-tolerant quantum computation is represented by a 3-dimensional structure called a BlockGraph. A BlockGraph represents the flow of a computation using logical qubits protected by a Surface Code. We can define our memory experiment using the existing library of pre-defined computations.

In [ ]:
blockgraph = memory(Basis.X)
blockgraph.view_as_html()

In TQEC, by convention, red is used to represent a Pauli-X while blue is used for a Pauli-Z. In this representation, the blue axis represent time (flowing from bottom to top) while the other two axes represent spatial dimensions.

We can relate the color of two faces on this cube to the specific memory experiment we are performing

* The bottom face (red) tells us we are preparing in the Pauli-X basis
* The bottom face (red) tells us we are measuring in the Pauli-X basis

In [ ]:
K = 1
scenario = f"Surface Code [d={2*K+1}]"
compiled = compile_block_graph(blockgraph)
scenarios[scenario] = compiled.generate_stim_circuit(k=K)

display(Markdown(f"[Open in Crumble ({scenario})]({compiled.generate_crumble_url(k=K, add_polygons=True)})"))

In [ ]:
simulate(scenarios, title=experiment, label="Case")

### Improving resilience to errors

By contrast with the physical error rate that can only be increased by improved engineering and manufacturing processes, the Surface Code enables us to increase the logical error rate by increasing the code distance (i.e. how many physical qubits are grouped into a single logical qubit).

In [ ]:
ler_per_curve_stats = start_simulation_using_sinter(
    blockgraph,
    ks = range(1, 5),
    ps = list(numpy.logspace(-4, -1, 10)),
    noise_model_factory=NoiseModel.uniform_depolarizing,
    manhattan_radius=2,
    convention=FIXED_BULK_CONVENTION,
    observables=blockgraph.find_correlation_surfaces(),
    max_shots=1_000_000,
    max_errors=5000,
    decoders=["pymatching"],
    split_observable_stats=False,
)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

zx_graph = blockgraph.to_zx_graph()
for i, obs in enumerate(blockgraph.find_correlation_surfaces()):
    main_stats = ler_per_curve_stats[i]

    fig, ax = plt.subplots()
    sinter.plot_error_rate(
        ax=ax,
        stats=main_stats,
        x_func=lambda stat: stat.json_metadata["p"],
        group_func=lambda stat: stat.json_metadata["d"],
    )
    ax.grid(which="both", axis="both")
    ax.legend()
    ax.loglog()
    ax.set_title(experiment)
    ax.set_xlabel("Physical error rate")
    ax.set_ylabel("Logical error rate")
    ax.set_ylim(10**-7.5, 10**0)

## How to perform a Hadamard ?

In [ ]:
experiment = "Hadamard experiment"
scenarios = {}

In [ ]:
scenario = "Physical"
circuit = stim.Circuit("""
    RZ 0
    TICK
    H 0
    TICK
    MX 0
    OBSERVABLE_INCLUDE(0) rec[-1]
""")
scenarios[scenario] = circuit
display(Markdown(f"[Open in Crumble ({scenario})]({circuit.to_crumble_url()})"))

In [ ]:
circuit.diagram("timeline-svg")

In [ ]:
simulate(scenarios, title=experiment, label="Case")

### How to protect it ?

In [ ]:
blockgraph = cnot(Basis.X)
blockgraph.view_as_html()

## How to perform a CNOT ?

In [ ]:
experiment = "X-basis CNOT experiment"
scenario = "Physical"

In [ ]:
circuit = stim.Circuit("""
    RX 0
    RZ 1
    TICK
    CX 0 1
    TICK
    MX 0 1
    OBSERVABLE_INCLUDE(0) rec[-1] rec[-2]
""")
scenarios[scenario] = circuit
display(Markdown(f"[Open in Crumble ({scenario})]({circuit.to_crumble_url()})"))

In [ ]:
simulate(scenarios, title=experiment, label="Case")

### And how to protect it ?

In [ ]:
blockgraph = cnot(Basis.X)
blockgraph.view_as_html()

In [ ]:
K = 1
scenario = f"Surface Code [d={2*K+1}]"
compiled = compile_block_graph(blockgraph)
scenarios[scenario] = compiled.generate_stim_circuit(k=K)

display(Markdown(f"[Open in Crumble ({scenario})]({compiled.generate_crumble_url(k=K, add_polygons=True)})"))

In [ ]:
simulate(scenarios, title=experiment, label="Case")

In [ ]:
ler_per_curve_stats = start_simulation_using_sinter(
    blockgraph,
    ks = range(1, 5),
    ps = list(numpy.logspace(-4, -1, 10)),
    noise_model_factory=NoiseModel.uniform_depolarizing,
    manhattan_radius=2,
    convention=FIXED_BULK_CONVENTION,
    observables=blockgraph.find_correlation_surfaces(),
    max_shots=1_000_000,
    max_errors=5000,
    decoders=["pymatching"],
    split_observable_stats=False,
)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

zx_graph = blockgraph.to_zx_graph()
for i, obs in enumerate(blockgraph.find_correlation_surfaces()):
    main_stats = ler_per_curve_stats[i]

    fig, ax = plt.subplots()
    sinter.plot_error_rate(
        ax=ax,
        stats=main_stats,
        x_func=lambda stat: stat.json_metadata["p"],
        group_func=lambda stat: stat.json_metadata["d"],
    )
    ax.grid(which="both", axis="both")
    ax.legend()
    ax.loglog()
    ax.set_title(experiment)
    ax.set_xlabel("Physical error rate")
    ax.set_ylabel("Logical error rate")
    ax.set_ylim(10**-7.5, 10**0)

## Extra - Pipeline (incl. ZX -> topologiq -> Blockgraph -> tqec -> stim)

## Extra - How to perform a Y-basis preparation/measurement ?

(See branch y-demo from Kabir)

## Extra - How to perform a T-gate ?

The ability to perform non-Clifford gates such as T and Toffoli gates determines the usefulness of a quantum computer. In the context of the Surface Code, it is not possible to directly (i.e. transversally) apply a T-gate on a logical state. The following figure presents the abstract circuit to perform this essential gate.

<div align="center">
  <img src="t-construct-circuit.png" width="640">
</div>

This circuit can in practice be realized as the following BlockGraph.

<div align="center">
  <img src="t-construct-blockgraph.png" width="640">
</div>

## How to determine the Threshold of the Surface Code ?

In [ ]:
# from math import log10
#
# thresholds: list[float] = []
# for obs in blockgraph.find_correlation_surfaces():
#     threshold, _ = binary_search_threshold(
#         blockgraph,
#         obs,
#         NoiseModel.uniform_depolarizing,
#         manhattan_radius=2,
#         minp=10**-5,
#         maxp=0.1,
#         convention=FIXED_BULK_CONVENTION,
#         max_shots=MAX_SHOTS,
#         max_errors=MAX_ERRORS,
#         decoders=["pymatching"],
#     )
#     thresholds.append(threshold)
#
# log10_thresholds = [log10(t) for t in thresholds]
# mint, maxt = min(log10_thresholds), max(log10_thresholds)
# log10_threshold_bounds = (mint - 0.2, maxt + 0.2)

In [ ]:
# fig, ax = plt.subplots()
# sinter.plot_error_rate(
#     ax=ax,
#     stats=ler_per_curve_stats,
#     x_func=lambda stat: stat.json_metadata["p"],
#     group_func=lambda stat: stat.json_metadata["d"],
# )
# xmin = 10 ** log10_threshold_bounds[0]
# xmax = 10 ** log10_threshold_bounds[1]
# # Note: the below values require prior knowledge about the values to look
# # for on the Y-axis.
# ymin, ymax = 1e-2, 3e-1
# plot_threshold_as_inset(
#     ax,
#     thres_stats,
#     # Note: ymax is **before** ymin because the y axis is inversed.
#     zoom_bounds=(xmin, ymax, xmax, ymin),
#     threshold=threshold,
#     inset_bounds=(0.5, 0.25, 0.4, 0.4),
# )
# ax.grid(which="both", axis="both")
# ax.legend()
# ax.loglog()
# ax.set_title("Z-basis memory error rate")
# ax.set_xlabel("Physical error rate (uniform depolarizing noise)")
# ax.set_ylabel("Logical error rate per shot")
# ax.set_ylim(10**-7.5, 10**0)

In [ ]:
# circuit = stim.Circuit("""
#     RX 0
#     RZ 1
#     TICK
#     CX 0 1
#     TICK
#     MPP Z0*Z1
#     OBSERVABLE_INCLUDE(0) rec[-1]
# """)
# scenarios["No QEC"] = circuit
#
# circuit.diagram("timeline-svg")

In [ ]:
# blockgraph = BlockGraph.from_dae_file("../../assets/logical_cnot.dae")
# surfaces = blockgraph.find_correlation_surfaces()